# Spambase & Wine Preprocessing

Pipeline: **download → inspect → split → quantile transform (fit on train only) → SMOTE if imbalanced (train only) → save**.

Each split is saved as a plain CSV with the target in the last column, so training code for any model can load it without re-running this notebook.

## 1. Setup

- The package below is needed for SMOTE. (For Colab)

In [1]:
%pip install -q imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\Myat\AppData\Local\Temp\claude\a--CMPE257-Project-DecisionTree-GradTree-Reproduction-\5e7b92bd-b88b-43cb-954a-a125da9f8905\scratchpad\venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import json
import urllib.request
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'preprocess':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = PROJECT_ROOT / 'dataset'          # raw files, e.g. dataset/wine/wine.data
PREPROCESS_DIR = PROJECT_ROOT / 'preprocess'    # outputs, e.g. preprocess/wine_output/
print('Project root:', PROJECT_ROOT)

Project root: A:\CMPE257\Project\DecisionTree_GradTree_Reproduction-


## 2. Settings

Make sure all datasets are processed the same way.

In [ ]:
# Defaults below match the official GradTree code (experiments_paper_gradtree/utilities/utilities_GDT.py)
SEEDS = [42 + i for i in range(10)]  # paper: data seed = 42 + trial, 10 trials
SEED = SEEDS[0]            # the split saved to preprocess/<name>_output/ (trial 0)
SAVE_ALL_SEEDS = False     # True: also save every seed to preprocess/<name>_output/seed_<seed>/

MATCH_PAPER_SPLIT = True   # True: 60/20/20 train/val/test, NOT stratified (as in the paper)
                           # False: proposal version, stratified 80/20, then 20% of train as validation
TEST_SIZE = 0.2
VAL_SIZE = 0.2             # paper: fraction of the FULL dataset; proposal: fraction of train

DROP_DUPLICATES = False    # paper keeps duplicates; True removes Spambase's duplicate rows
BALANCE_THRESHOLD = 0.25   # paper: SMOTE if smallest class share <= 0.25 / (num_classes - 1)
QUANTILE_NOISE = 1e-3      # paper adds tiny noise to train data before fitting the quantile transform
QT_OUTPUT = 'normal'       # paper uses output_distribution='normal'

## 3. Download the data

The original UCI `.data` files are downloaded once and kept in `dataset/<name>/`, so everyone preprocesses exactly the same raw data and later runs don't depend on UCI being online. If UCI is unavailable, the copy in the GradTree repository is used instead. The files have no header row, so the column names below are the ones UCI lists for each dataset.

In [4]:
UCI_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases'
MIRROR_URL = 'https://raw.githubusercontent.com/s-marton/GradTree/master/experiments_paper_gradtree/real_world_datasets'

SPAMBASE_WORDS = [
    'make', 'address', 'all', '3d', 'our', 'over', 'remove', 'internet', 'order', 'mail',
    'receive', 'will', 'people', 'report', 'addresses', 'free', 'business', 'email', 'you', 'credit',
    'your', 'font', '000', 'money', 'hp', 'hpl', 'george', '650', 'lab', 'labs',
    'telnet', '857', 'data', '415', '85', 'technology', '1999', 'parts', 'pm', 'direct',
    'cs', 'meeting', 'original', 'project', 're', 'edu', 'table', 'conference',
]
SPAMBASE_COLUMNS = (
    [f'word_freq_{word}' for word in SPAMBASE_WORDS]
    + [f'char_freq_{char}' for char in ';([!$#']
    + ['capital_run_length_average', 'capital_run_length_longest', 'capital_run_length_total', 'Class']
)
WINE_COLUMNS = [
    'class',  # wine.data has the label first
    'Alcohol', 'Malicacid', 'Ash', 'Alcalinity_of_ash', 'Magnesium', 'Total_phenols', 'Flavanoids',
    'Nonflavanoid_phenols', 'Proanthocyanins', 'Color_intensity', 'Hue', '0D280_0D315_of_diluted_wines', 'Proline',
]

# name: (download URLs, column names, target column)
SOURCES = {
    'spambase': ([f'{UCI_URL}/spambase/spambase.data', f'{MIRROR_URL}/Spambase/spambase.data'],
                 SPAMBASE_COLUMNS, 'Class'),
    'wine': ([f'{UCI_URL}/wine/wine.data', f'{MIRROR_URL}/Wine/wine.data'],
             WINE_COLUMNS, 'class'),
}

def download(urls, path):
    if path.exists() and path.stat().st_size > 0:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    errors = []
    for url in urls:
        try:
            urllib.request.urlretrieve(url, path)
            return
        except Exception as exc:
            errors.append(f'{url}: {exc}')
    raise RuntimeError(f'Could not download {path.name}:\n' + '\n'.join(errors))

DATASETS, TARGETS = {}, {}
for name, (urls, columns, target) in SOURCES.items():
    path = DATASET_DIR / name / f'{name}.data'
    download(urls, path)
    data = pd.read_csv(path, names=columns)
    DATASETS[name] = (data.drop(columns=target), data[target])
    TARGETS[name] = target
    print(f'{name}: {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size:,} bytes)')

spambase: dataset\spambase\spambase.data (702,942 bytes)


wine: dataset\wine\wine.data (10,782 bytes)


## 4. Inspect before changing anything

Expected: Spambase 4,601 × 57, Wine 178 × 13, no missing values, no categorical columns.

In [5]:
EXPECTED_SHAPES = {'spambase': (4601, 57), 'wine': (178, 13)}

for name, (X, y) in DATASETS.items():
    print(f'=== {name} ===')
    print('Shape:', X.shape)
    print('Missing values:', int(X.isna().sum().sum()), '| missing targets:', int(y.isna().sum()))
    print('Non-numeric columns:', list(X.select_dtypes(exclude='number').columns))
    print('Duplicate rows (features + label):', int(pd.concat([X, y], axis=1).duplicated().sum()))
    print('Class balance:')
    print(y.value_counts(normalize=True).round(3).to_string())
    print()
    assert X.shape == EXPECTED_SHAPES[name]
    assert not X.isna().any().any() and not y.isna().any()

=== spambase ===
Shape: (4601, 57)
Missing values: 0 | missing targets: 0
Non-numeric columns: []
Duplicate rows (features + label): 391
Class balance:
Class
0    0.606
1    0.394

=== wine ===
Shape: (178, 13)
Missing values: 0 | missing targets: 0
Non-numeric columns: []
Duplicate rows (features + label): 0
Class balance:
class
2    0.399
1    0.331
3    0.270



In [6]:
DATASETS['wine'][0].describe().T  # change to 'spambase' to look at Spambase

,count,mean,std,min,25%,50%,75%,max
Alcohol,178.0,13.000618,0.811827,11.03,12.3625,13.050,13.6775,14.83
Malicacid,178.0,2.336348,1.117146,0.74,1.6025,1.865,3.0825,5.80
Ash,178.0,2.366517,0.274344,1.36,2.2100,2.360,2.5575,3.23
Alcalinity_of_ash,178.0,19.494944,3.339564,10.60,17.2000,19.500,21.5000,30.00
Magnesium,178.0,99.741573,14.282484,70.00,88.0000,98.000,107.0000,162.00
Total_phenols,178.0,2.295112,0.625851,0.98,1.7425,2.355,2.8000,3.88
Flavanoids,178.0,2.029270,0.998859,0.34,1.2050,2.135,2.8750,5.08
Nonflavanoid_phenols,178.0,0.361854,0.124453,0.13,0.2700,0.340,0.4375,0.66
Proanthocyanins,178.0,1.590899,0.572359,0.41,1.2500,1.555,1.9500,3.58
Color_intensity,178.0,5.058090,2.318286,1.28,3.2200,4.690,6.2000,13.00


## 5. Clean

In [7]:
def clean(X, y):
    X, y = X.copy(), y.copy()
    if DROP_DUPLICATES:
        keep = ~pd.concat([X, y], axis=1).duplicated()
        X, y = X[keep], y[keep]
    return X.reset_index(drop=True), y.reset_index(drop=True)

for name, (X, y) in DATASETS.items():
    Xc, yc = clean(X, y)
    DATASETS[name] = (Xc, yc)
    print(f'{name}: {X.shape[0]} -> {Xc.shape[0]} rows after cleaning')

spambase: 4601 -> 4601 rows after cleaning
wine: 178 -> 178 rows after cleaning


## 6. Shared pipeline: split → transform → SMOTE → encode

Every transform is **fit on the training set only**, then applied to validation and test. Validation and test data are never resampled.

In [8]:
def finish_pipeline(X, y, seed):
    X = X.to_numpy(dtype=float)

    # Encode labels to 0..K-1 (Wine uses 1/2/3; PyTorch cross-entropy needs 0-based)
    le = LabelEncoder().fit(y)
    y = le.transform(y)

    if MATCH_PAPER_SPLIT:
        # Paper: test and validation are each 20% of the full data, no stratification
        n = len(X)
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=int(n * TEST_SIZE), random_state=seed)
        X_tr, X_valid, y_tr, y_valid = train_test_split(
            X_tr, y_tr, test_size=int(n * VAL_SIZE), random_state=seed)
    else:
        # Proposal: stratified 80/20, then 20% of train for validation
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)
        X_tr, X_valid, y_tr, y_valid = train_test_split(
            X_tr, y_tr, test_size=VAL_SIZE, stratify=y_tr, random_state=seed)

    # Quantile transform: fit on (noisy copy of) train only, then transform the real values.
    # The noise breaks ties in features with many repeated values, e.g. Spambase's zeros.
    X_fit = X_tr.copy()
    if QUANTILE_NOISE > 0:
        rng = np.random.RandomState(seed)
        stds = X_fit.std(axis=0, keepdims=True)
        X_fit += QUANTILE_NOISE / np.maximum(stds, QUANTILE_NOISE) * rng.randn(*X_fit.shape)
    qt = QuantileTransformer(n_quantiles=min(1000, len(X_tr)), output_distribution=QT_OUTPUT,
                             subsample=None, random_state=seed)
    qt.fit(X_fit)
    X_tr, X_valid, X_te = qt.transform(X_tr), qt.transform(X_valid), qt.transform(X_te)

    # SMOTE on the training set only, using the paper's rule
    n_train = len(y_tr)
    n_classes = len(np.unique(y_tr))
    minority_share = np.bincount(y_tr).min() / len(y_tr)
    smote_threshold = BALANCE_THRESHOLD / (n_classes - 1)
    smote_used = minority_share <= smote_threshold
    if smote_used:
        X_tr, y_tr = SMOTE(random_state=seed).fit_resample(X_tr, y_tr)

    return dict(X_train=X_tr, X_valid=X_valid, X_test=X_te,
                y_train=y_tr, y_valid=y_valid, y_test=y_te,
                label_encoder=le, quantile_transformer=qt, train_rows_before_smote=n_train,
                minority_share=minority_share, smote_threshold=smote_threshold, smote_used=smote_used)

## 7. Save processed datasets and fitted transformers

Each dataset is saved to `preprocess/<name>_output/`:

- `<name>_train.csv`, `<name>_valid.csv`, `<name>_test.csv`: transformed features, with the target (encoded to 0..K-1) as the last column. CSV can be read by any library, and keeping the target last means one loader works for every dataset.
- `<name>_preprocessor.joblib`: the fitted quantile transformer and label encoder. Their `inverse_transform` maps features and predictions back to the original values, which helps with error analysis.
- `<name>_preprocessing_summary.json`: row counts and SMOTE details, to check a run without opening the CSVs.

Only the `SEED` split (trial 0) is saved by default, which keeps the files small enough to commit. Set `SAVE_ALL_SEEDS = True` to also write every seed to `seed_<seed>/` for 10-trial experiments.

In [9]:
def save_outputs(name, X, y, seed, out_dir):
    d = finish_pipeline(X, y, seed)
    out_dir.mkdir(parents=True, exist_ok=True)
    feature_names = list(X.columns)
    target = TARGETS[name]
    le = d['label_encoder']

    def save_split(part, features, labels):
        output = pd.DataFrame(features, columns=feature_names)
        output[target] = np.asarray(labels, dtype=np.int8)
        output.to_csv(out_dir / f'{name}_{part}.csv', index=False)

    save_split('train', d['X_train'], d['y_train'])
    save_split('valid', d['X_valid'], d['y_valid'])
    save_split('test', d['X_test'], d['y_test'])

    # No categorical features, so no target encoder; the keys stay so every preprocessor file has the same fields
    joblib.dump({
        'encoder': None,
        'quantile_transformer': d['quantile_transformer'],
        'label_encoder': le,
        'feature_names': feature_names,
        'categorical_features': [],
        'numeric_features': feature_names,
        'target': target,
        'random_seed': seed,
        'quantile_noise': QUANTILE_NOISE,
    }, out_dir / f'{name}_preprocessor.joblib')

    def counts(labels):
        return {int(label): int(count) for label, count in enumerate(np.bincount(labels))}

    summary = {
        'source_rows': int(len(X)),
        'feature_count': int(X.shape[1]),
        'target': target,
        'classes': [str(c) for c in le.classes_],  # original label of encoded class 0, 1, ...
        'original_target_counts': counts(le.transform(y)),
        'missing_counts': {column: int(count) for column, count in X.isna().sum().items() if count},
        'split_rows': {'train': int(d['train_rows_before_smote']),
                       'valid': int(len(d['y_valid'])), 'test': int(len(d['y_test']))},
        'train_rows_after_smote': int(len(d['y_train'])),
        'minority_ratio_before_smote': float(d['minority_share']),
        'smote_threshold': float(d['smote_threshold']),
        'smote_applied': bool(d['smote_used']),
        'train_target_counts_after_smote': counts(d['y_train']),
        'random_seed': seed,
    }
    (out_dir / f'{name}_preprocessing_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return summary

for name, (X, y) in DATASETS.items():
    out_dir = PREPROCESS_DIR / f'{name}_output'
    summary = save_outputs(name, X, y, SEED, out_dir)
    print(f'=== {name} ===')
    print(json.dumps(summary, indent=2))
    if SAVE_ALL_SEEDS:
        for seed in SEEDS:
            save_outputs(name, X, y, seed, out_dir / f'seed_{seed}')
        print(f'Also saved {len(SEEDS)} seeds to {out_dir.relative_to(PROJECT_ROOT)}/seed_<seed>/')
    print('Saved files:')
    for path in sorted(out_dir.glob('*.*')):
        print(f'- {path.name} ({path.stat().st_size:,} bytes)')
    print()

=== spambase ===
{
  "source_rows": 4601,
  "feature_count": 57,
  "target": "Class",
  "classes": [
    "0",
    "1"
  ],
  "original_target_counts": {
    "0": 2788,
    "1": 1813
  },
  "missing_counts": {},
  "split_rows": {
    "train": 2761,
    "valid": 920,
    "test": 920
  },
  "train_rows_after_smote": 2761,
  "minority_ratio_before_smote": 0.38500543281419775,
  "smote_threshold": 0.25,
  "smote_applied": false,
  "train_target_counts_after_smote": {
    "0": 1698,
    "1": 1063
  },
  "random_seed": 42
}
Saved files:
- spambase_preprocessing_summary.json (532 bytes)
- spambase_preprocessor.joblib (465,933 bytes)
- spambase_test.csv (1,071,823 bytes)
- spambase_train.csv (3,215,620 bytes)
- spambase_valid.csv (1,071,792 bytes)

=== wine ===
{
  "source_rows": 178,
  "feature_count": 13,
  "target": "class",
  "classes": [
    "1",
    "2",
    "3"
  ],
  "original_target_counts": {
    "0": 59,
    "1": 71,
    "2": 48
  },
  "missing_counts": {},
  "split_rows": {
    "tra

## 8. Sanity checks

1. Class proportions should be similar across train/valid/test (exactly equal only with stratified splits).
2. Transformed training features should look roughly normal (mean ≈ 0, std ≈ 1 if `QT_OUTPUT='normal'`).
3. A quick CART should get roughly 90% accuracy on Spambase and above 85% on Wine. Much worse usually means a pipeline bug.

`load_split` reads the saved CSVs the way training code would, so these checks also test the saved files.

In [10]:
def load_split(name, out_dir=None):
    out_dir = out_dir or PREPROCESS_DIR / f'{name}_output'
    d = {}
    for part in ['train', 'valid', 'test']:
        frame = pd.read_csv(out_dir / f'{name}_{part}.csv')
        d[f'X_{part}'] = frame.iloc[:, :-1].to_numpy()
        d[f'y_{part}'] = frame.iloc[:, -1].to_numpy()
    return d

for name, (X, y) in DATASETS.items():
    d = load_split(name)
    print(f'=== {name} (saved split, seed {SEED}) ===')
    for part in ['train', 'valid', 'test']:
        labels = d[f'y_{part}']
        print(f'  {part:5s} rows {len(labels):4d} | class shares: {np.round(np.bincount(labels) / len(labels), 3)}')
    print(f'  train feature mean {d["X_train"].mean():.2f}, std {d["X_train"].std():.2f}')

    # CART over every seed (not saved), to check the pipeline isn't tuned to one split
    accs, f1s = [], []
    for seed in SEEDS:
        s = finish_pipeline(X, y, seed)
        pred = DecisionTreeClassifier(random_state=seed).fit(s['X_train'], s['y_train']).predict(s['X_test'])
        accs.append(accuracy_score(s['y_test'], pred))
        f1s.append(f1_score(s['y_test'], pred, average='macro'))
    print(f'  CART over {len(SEEDS)} seeds: accuracy {np.mean(accs):.3f} ± {np.std(accs):.3f}, '
          f'macro F1 {np.mean(f1s):.3f} ± {np.std(f1s):.3f}\n')

=== spambase (saved split, seed 42) ===
  train rows 2761 | class shares: [0.615 0.385]
  valid rows  920 | class shares: [0.609 0.391]
  test  rows  920 | class shares: [0.576 0.424]
  train feature mean 0.06, std 0.67


  CART over 10 seeds: accuracy 0.905 ± 0.011, macro F1 0.901 ± 0.011

=== wine (saved split, seed 42) ===
  train rows  108 | class shares: [0.315 0.407 0.278]
  valid rows   35 | class shares: [0.343 0.371 0.286]
  test  rows   35 | class shares: [0.371 0.4   0.229]
  train feature mean -0.01, std 1.16


  CART over 10 seeds: accuracy 0.880 ± 0.051, macro F1 0.881 ± 0.056



## Loading the splits later

In the GradTree or CART training code, every dataset loads the same way: features first, target last.
```python
import pandas as pd

train = pd.read_csv('preprocess/wine_output/wine_train.csv')
X_train, y_train = train.iloc[:, :-1].to_numpy(), train.iloc[:, -1].to_numpy()
```
Labels are already 0..K-1. `joblib.load('preprocess/wine_output/wine_preprocessor.joblib')['label_encoder']` maps them back to the original labels.